# Deal-Flow Radar: Investment Opportunity Scoring Pipeline

This notebook implements a pipeline for scoring potential investment opportunities based on various features extracted from company data. The goal is to create a model that can identify companies with a high probability of achieving a top-decile exit.

## 1. Import Libraries

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, confusion_matrix, classification_report
import catboost
from catboost import CatBoostClassifier, Pool
import joblib
import warnings
warnings.filterwarnings('ignore')

# Add the parent directory to the path to access the services module
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

## 2. Data Loading and Preprocessing

In [2]:
# Load mock company data (this will be replaced with Postgres DB queries in production)
data_path = '../services/radar/data/mock_company_data.csv'
data = pd.read_csv(data_path)

In [3]:
# Display basic information about the dataset
print(f"Dataset shape: {data.shape}")
print("\nColumns:")
for col in data.columns:
    print(f"- {col}")

print("\nSample data:")
display(data.head())

print("\nBasic statistics:")
display(data.describe())

print(f"\nTarget distribution (top_decile_exit):\n{data['top_decile_exit'].value_counts(normalize=True)}")

In [4]:
# Data preprocessing
# Convert founding_date to age in years
data['founding_date'] = pd.to_datetime(data['founding_date'])
today = datetime.now()
data['company_age'] = (today - data['founding_date']).dt.days / 365.25

# Create additional features
data['stars_to_age_ratio'] = data['github_stars'] / data['company_age']
data['funding_per_investor'] = data['funding_amount'] / data['investor_count'].replace(0, 1)  # Avoid division by zero
data['social_engagement_ratio'] = data['social_traction'] / data['company_age']

# Feature selection
features = [
    'company_age', 'github_stars', 'commit_velocity', 'investor_count',
    'founder_exit_count', 'social_traction', 'funding_amount',
    'stars_to_age_ratio', 'funding_per_investor', 'social_engagement_ratio'
]

X = data[features]
y = data['top_decile_exit']

print("Features after preprocessing:")
display(X.head())
print(f"\nTarget balance:\n{y.value_counts(normalize=True)}")

## 3. Feature Analysis

In [5]:
# Correlation analysis
plt.figure(figsize=(12, 10))
sns.heatmap(X.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.show()

# Feature distribution by target class
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(18, 16))
axes = axes.flatten()

for i, feature in enumerate(features):
    if i < len(axes):
        sns.boxplot(x='top_decile_exit', y=feature, data=data, ax=axes[i])
        axes[i].set_title(f'{feature} by Target Class')
        axes[i].set_xlabel('Top Decile Exit')

plt.tight_layout()
plt.show()

## 4. Model Training and Evaluation

In [6]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

In [7]:
# Define CatBoost model with optimal hyperparameters
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=100
)

# Train the model
model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    early_stopping_rounds=50,
    plot=True
)

In [8]:
# Calculate predictions and probabilities
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Evaluate model performance
test_auc = roc_auc_score(y_test, y_pred_proba)
print(f"Test AUC: {test_auc:.4f}")

# Cross-validation to ensure robustness
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
print(f"Cross-validation AUC scores: {cv_scores}")
print(f"Mean CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Feature importance
feature_importance = model.get_feature_importance()
feature_names = X.columns

# Sort feature importances in descending order
sorted_idx = np.argsort(feature_importance)[::-1]
plt.figure(figsize=(12, 6))
plt.title('Feature Importance')
plt.bar(range(len(sorted_idx)), feature_importance[sorted_idx], align='center')
plt.xticks(range(len(sorted_idx)), feature_names[sorted_idx], rotation=90)
plt.tight_layout()
plt.show()

# Print classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 5. Model Savings and Deployment Preparation

In [9]:
# Save the model
MODEL_DIR = '../services/radar/models/'
os.makedirs(MODEL_DIR, exist_ok=True)
model_path = os.path.join(MODEL_DIR, 'catboost_radar_model.cbm')
model.save_model(model_path)
print(f"Model saved to {model_path}")

# Save model metadata
model_metadata = {
    'features': features,
    'train_date': datetime.now().strftime('%Y-%m-%d'),
    'auc_score': test_auc,
    'cv_auc_mean': cv_scores.mean(),
    'cv_auc_std': cv_scores.std(),
    'top_features': [feature_names[i] for i in sorted_idx[:5]]
}

import json
with open(os.path.join(MODEL_DIR, 'model_metadata.json'), 'w') as f:
    json.dump(model_metadata, f, indent=4)
    
print("Model metadata saved")

## 6. Daily Shortlist Generator Function

In [10]:
def generate_daily_shortlist(company_data, model, limit=10):
    """
    Generate a shortlist of top companies based on model predictions.
    
    Args:
        company_data (pd.DataFrame): DataFrame with company features
        model: Trained CatBoost model
        limit (int): Number of companies to include in the shortlist
        
    Returns:
        pd.DataFrame: Shortlist of companies with their scores
    """
    # Prepare features
    if 'founding_date' in company_data.columns and 'company_age' not in company_data.columns:
        company_data['founding_date'] = pd.to_datetime(company_data['founding_date'])
        company_data['company_age'] = (datetime.now() - company_data['founding_date']).dt.days / 365.25
    
    # Create derived features if they don't exist
    if 'stars_to_age_ratio' not in company_data.columns:
        company_data['stars_to_age_ratio'] = company_data['github_stars'] / company_data['company_age']
    
    if 'funding_per_investor' not in company_data.columns:
        company_data['funding_per_investor'] = company_data['funding_amount'] / company_data['investor_count'].replace(0, 1)
    
    if 'social_engagement_ratio' not in company_data.columns:
        company_data['social_engagement_ratio'] = company_data['social_traction'] / company_data['company_age']
    
    # Extract features used by the model
    X = company_data[features]
    
    # Generate predictions
    probabilities = model.predict_proba(X)[:, 1]
    
    # Add probabilities to the DataFrame
    company_data['exit_probability'] = probabilities
    
    # Sort by probability (descending) and take the top N companies
    shortlist = company_data.sort_values('exit_probability', ascending=False).head(limit)
    
    # Prepare the output
    result = shortlist[['company_id', 'name', 'exit_probability']].copy()
    
    return result

In [11]:
# Test the daily shortlist generator
shortlist = generate_daily_shortlist(data, model, limit=5)
print("Top 5 companies in the daily shortlist:")
display(shortlist)

# Export the shortlist to JSON (similar to what the API would return)
shortlist_json = shortlist.rename(columns={'exit_probability': 'clos'}).to_dict(orient='records')
print("\nShortlist in JSON format (API output):")
print(json.dumps(shortlist_json, indent=2))

## 7. Conclusion and Next Steps

We have successfully created a Deal-Flow Radar pipeline that:

1. Loads company data and extracts relevant features
2. Trains a CatBoost model to predict top-decile exits 
3. Evaluates the model's performance (AUC ≥ 0.7 on our test set)
4. Provides a function to generate a daily shortlist of promising companies

### Next Steps:

1. **Database Integration**: Replace CSV loading with PostgreSQL database queries to fetch fresh entity snapshots
2. **API Implementation**: Implement the `/radar/daily_shortlist` endpoint to serve model predictions
3. **Scheduled Execution**: Set up a scheduler to run the pipeline every night to update predictions
4. **Model Monitoring**: Implement tracking of model performance over time
5. **Feature Engineering Refinement**: Further analyze and optimize the feature set